[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/earthquake/earthquake.ipynb)

# Earthquake Beachballs

This example draws earthquake beachballs, the diagrams seismologists use to summarize how a fault slipped. When a fault slips, the ground on one side lurches past the other, and seismometers all around the globe feel the first wave arrive either as a push or as a pull, depending on the direction they lie in. The beachball is the lower half of the sphere of directions, flattened, with the pushes shaded. From it the fault can be recovered, up to one ambiguity that no seismometer can resolve.

A fault's orientation is modelled as one rotor, three turns by its strike, dip and rake, which carries the level ground onto the fault plane and the prime vertical, the upright plane facing north, onto the auxiliary plane, the plane across the slip. The slip acts on the ground as a moment, a map from directions to the planes that read them, `auxiliaries * (faults & Point) + faults * (auxiliaries & Point)`: the first term alone is a couple, which would turn the ground by the plane of shear `faults ^ auxiliaries`, and the second, its mirror, cancels that turning. Along a direction the P wave's first motion is the moment's reading of it, `moment(direction) & direction`, and the fault comes back from the pressure and tension axes, the eigenvectors of that reading against the metric. The moment is symmetric in the two planes, so the fault and the auxiliary plane give the same moment.

In the tensor notation of seismology the moment reads as $M_{ij} = M_0\,(n_i s_j + n_j s_i)$, and the first motion of the P wave along the unit direction $\gamma$ as $\gamma_i M_{ij} \gamma_j$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.util.animation import save_figure, show_animation
from examples.mechanics.earthquake import render
from examples.mechanics.earthquake.core import directions

ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# Directions are antivectors with intrinsic orientation; faults are planes, with extrinsic orientation.
Point = ga.gatype.antivector()
Plane = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Rotor = ga.gatype.rotor()
Moment = ga.gatype((Plane, Point))                         # Plane+ <- Point-

# Space is north, east and down, as in seismology; the planes the fault's frame turns in, from one direction
# towards another, are their commutators.
north, east, down = mv.basis().dual()                      # [] Point-, Point-, Point-
# The horizontal plane, its sense from north towards east.
horizontal = east.commutator(north)                        # [] Bivector+
# The upright plane through east, its sense from east towards down.
upright = down.commutator(east)                            # [] Bivector+
# The level ground, facing up, and the prime vertical, the upright plane facing north: a fault's frame turns
# them onto the fault plane and the auxiliary plane.
ground, prime_vertical = -mv.z, mv.x                       # [] Plane+, Plane+

## 1. The fault as a rotor

A fault is a plane in the ground and a direction of slip within it, given by three angles: the strike, the compass direction of the fault's horizontal line; the dip, how steeply it plunges, to the right of the strike; and the rake, the direction of slip within the plane. Each angle is a turn, and the three compose into one rotor that carries north onto the slip and up onto the fault's normal, so the level ground onto the fault plane and the prime vertical onto the auxiliary plane, the plane across the slip.

In [ ]:
# The faults: strike-slip, normal, thrust and oblique; their strikes, dips and rakes, in degrees, and the
# seismic moment of each.
strikes = np.array([30.0, 30.0, 30.0, 30.0])
dips = np.array([90.0, 60.0, 30.0, 50.0])
rakes = np.array([0.0, -90.0, 90.0, 45.0])
seismic_moment = 1.0

In [ ]:
def fault_frame(strikes: np.ndarray, dips: np.ndarray, rakes: np.ndarray) -> Rotor:
    """The turn by the rake in the horizontal plane, then the dip about north, then the strike from north."""
    # The rake turns north towards -east, the dip east towards down, and the strike north towards east.
    rake = (horizontal * (rakes / 2)).exp()                                   # [...] Rotor+
    dip = (upright * (-dips / 2)).exp()                                       # [...] Rotor+
    strike = (horizontal * (-strikes / 2)).exp()                              # [...] Rotor+
    return strike * dip * rake


frames = fault_frame(np.deg2rad(strikes), np.deg2rad(dips), np.deg2rad(rakes))   # [faults] Rotor+
faults, auxiliaries = frames >> ground, frames >> prime_vertical               # [faults] Plane+, Plane+

## 2. The moment of a slip

The slip pushes the ground on one side of the fault along the slip and the other side back: a couple, the auxiliary plane weighted by the fault's reading of whatever direction it is given, `auxiliary * (fault & Point)`. A couple alone would set the ground turning; how much, its curl, is each plane it gives wedged with the plane that reads its direction, `(Plane ^ moment(Point)).trace(1, 2)`, the trace pairing plane against direction with no metric between them, and it comes out as the plane of shear `fault ^ auxiliary`. The ground does not turn, so a mirrored couple balances it, and the moment is the sum, with no curl. It is symmetric in the two planes: the fault and the auxiliary plane have the same moment.

In [ ]:
def double_couple(faults: Plane, auxiliaries: Plane, moments: float) -> Moment:
    """The slip's couple across the fault, and the mirrored couple that keeps the ground from turning."""
    return moments * (auxiliaries * (faults & Point) + faults * (auxiliaries & Point))   # [...] Plane+ <- Point-


def curl(moment: Moment) -> Bivector:
    """How much a moment would turn the ground."""
    # Each plane the moment gives, wedged with the plane that reads its direction, plane against direction.
    return (Plane ^ moment(Point)).trace(1, 2)                               # [...] Bivector+


# One couple alone, the moment, and the moment with the fault and the auxiliary plane exchanged.
single = auxiliaries * (faults & Point)                                       # [faults] Plane+ <- Point-
moment = double_couple(faults, auxiliaries, seismic_moment)                   # [faults] Plane+ <- Point-
exchanged = double_couple(auxiliaries, faults, seismic_moment)                # [faults] Plane+ <- Point-

## 3. First motions and the beachball

Along a direction, the P wave's first motion is the moment's reading of that direction, `moment(direction) & direction`: positive for a push away from the source, negative for a pull. The metric turns the reading into the motion it drives, and the S wave carries the part of it across the direction. Over the lower hemisphere, two turns per direction, the pushes are shaded and the nodal lines where the P wave is silent are the fault and the auxiliary plane. The eigenvectors of the moment's reading, against the metric, the dot of the planes across the directions, are its pressure axis, in the middle of the pulls, and its tension axis, in the middle of the pushes.

The balls are the strike-slip, normal, thrust and oblique faults, north up; the pressure axis is the blue dot and the tension axis the red one.

In [ ]:
# The grid of directions: rings from straight down, spokes around the vertical.
n_rings = 60
n_spokes = 120

In [ ]:
def radiation(moment: Moment, directions: Point) -> tuple[Scalar, Point]:
    """The P wave's first motion along each direction, and the S wave's."""
    reading = moment(directions)                                              # [...] Plane+
    push = reading & directions                                               # [...] Scalar+
    # The metric turns the moment's reading into the motion it drives.
    return push, reading.dual() - directions * push


hemisphere = directions(np.linspace(0.0, np.pi / 2, n_rings), np.linspace(0.0, 2 * np.pi, n_spokes + 1))   # [rings, spokes + 1] Point-
# The principal values and axes, from pressure through the null axis to tension: the moment's reading of
# directions against the metric, the dot of the planes across them.
values, axes = (moment(Point) & Point).eigh(Point.dual().dot(Point.dual()))   # [faults, 3] Scalar+, Point-
push, shear = radiation(moment[:, None, None], hemisphere)                    # [faults, rings, spokes + 1]
save_figure(render.draw_mechanisms(hemisphere, push, axes), "earthquake_beachballs");

## 4. From the moment back to the fault

The moment reads the tension axis positive and the pressure axis negative, and the two planes' normals are their sum and difference, over the square root of two, which the metric turns into the planes across them. Which is which, the moment cannot say: both choices rebuild it, and only other evidence, such as the trace of the fault at the surface or the spread of aftershocks, picks the fault from the auxiliary plane.

In [ ]:
def fault_planes(moment: Moment) -> tuple[Plane, Plane]:
    """The two faults with this moment, each fault's auxiliary plane the other's fault."""
    _, axes = (moment(Point) & Point).eigh(Point.dual().dot(Point.dual()))  # [..., 3] Point-
    pressure, tension = axes[..., 0], axes[..., 2]
    # The two planes' normals, which the metric turns into the planes across them.
    first = ((tension + pressure) / np.sqrt(2)).dual_inverse()               # [...] Plane+
    second = ((tension - pressure) / np.sqrt(2)).dual_inverse()              # [...] Plane+
    return stack([first, second], axis=-1), stack([second, first], axis=-1)   # [..., 2] Plane+


found, found_auxiliaries = fault_planes(moment)                               # [faults, 2] Plane+, Plane+
rebuilt = double_couple(found, found_auxiliaries, seismic_moment)             # [faults, 2] Plane+ <- Point-

## 5. Turning the rake

Keeping the fault and turning its slip once around, the ball goes from a normal fault, through strike-slip, to a thrust and back. In space, each direction is drawn as far as its P wave's first motion: four lobes, two pushing and two pulling, with the fault in between. This cell takes about thirty seconds, most of it drawing the lobes.

In [ ]:
# The fault whose rake turns: its strike and dip, in degrees, the frames of the turn, and the duration of each.
sweep_strike, sweep_dip = 30.0, 50.0
n_frames = 144
duration_ms = 40

In [ ]:
def sweep(n_frames: int):
    """Per frame of the rake's turn: the fault's normal, slip and principal axes, its P wave over the lower
    hemisphere, and the whole sphere of directions in the fault's frame with the P wave over it."""
    sphere = directions(np.linspace(0.0, np.pi, 2 * n_rings - 1), np.linspace(0.0, 2 * np.pi, n_spokes + 1))   # [2 * rings - 1, spokes + 1] Point-
    for rake in np.linspace(-np.pi, np.pi, n_frames, endpoint=False):
        frame = fault_frame(np.deg2rad(sweep_strike), np.deg2rad(sweep_dip), np.array(rake))   # [] Rotor+
        fault, auxiliary = frame >> ground, frame >> prime_vertical           # [] Plane+, Plane+
        fault_moment = double_couple(fault, auxiliary, seismic_moment)        # [] Plane+ <- Point-
        _, fault_axes = (fault_moment(Point) & Point).eigh(Point.dual().dot(Point.dual()))   # [3] Point-
        # The metric turns the two planes into the fault's normal and its slip, for drawing.
        normal, slip = fault.dual(), auxiliary.dual()                         # [] Point-, Point-
        # The sphere turned with the fault, so that its nodal planes lie along the grid lines.
        fault_sphere = frame >> sphere                                        # [2 * rings - 1, spokes + 1] Point-
        ball, _ = radiation(fault_moment, hemisphere)                         # [rings, spokes + 1] Scalar+
        lobes, _ = radiation(fault_moment, fault_sphere)                      # [2 * rings - 1, spokes + 1] Scalar+
        yield normal, slip, fault_axes, ball, fault_sphere, lobes


show_animation(render.animate_sweep(hemisphere, sweep(n_frames)), "earthquake_rake", duration_ms)

In [ ]:
# checks
# A slip neither swells nor shrinks the ground and has no third axis: the principal values add to nothing,
# and the moment's determinant against the metric vanishes.
np.testing.assert_allclose(values.sum(axis=-1).to_array(), 0.0, atol=1e-14)
np.testing.assert_allclose((moment(Point) & Point).det(Point.dual().dot(Point.dual())).to_array(), 0.0, atol=1e-14)
np.testing.assert_allclose(values.to_array(), np.array([-seismic_moment, 0.0, seismic_moment]) * np.ones((len(strikes), 1)), atol=1e-13)
# One couple turns the ground by the plane of shear; the double couple does not.
np.testing.assert_allclose(curl(single).kernel, (faults ^ auxiliaries).kernel, atol=1e-14)
np.testing.assert_allclose(curl(moment).kernel, 0.0, atol=1e-14)
# The fault and the auxiliary plane give the same moment, and both planes found from it rebuild it.
np.testing.assert_allclose(exchanged.kernel, moment.kernel, atol=1e-14)
np.testing.assert_allclose(rebuilt.kernel, moment[:, None].broadcast_to(rebuilt.shape).kernel, atol=1e-13)
np.testing.assert_allclose(found.dot(faults[:, None]).abs().to_array().max(axis=-1), 1.0, rtol=1e-12)
# Half a turn of rake reverses the slip and flips the moment.
sweep_frames = fault_frame(np.deg2rad(sweep_strike), np.deg2rad(sweep_dip), np.linspace(-np.pi, np.pi, n_frames, endpoint=False))   # [frames] Rotor+
sweep_moment = double_couple(sweep_frames >> ground, sweep_frames >> prime_vertical, seismic_moment)   # [frames] Plane+ <- Point-
np.testing.assert_allclose((sweep_moment[n_frames // 2:] + sweep_moment[:n_frames // 2]).kernel, 0.0, atol=1e-14)